In [1]:
import os
import sys
from typing import Tuple
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# CUSTOM EXCEPTION HIERARCHY
class CreditSystemError(Exception):
    """Base exception for credit assessment workflow."""
    pass


class DatasetError(CreditSystemError):
    """Raised when there is an issue loading or cleaning the dataset."""
    pass


class ModelNotFittedError(CreditSystemError):
    """Raised when inference is called prior to model fitting."""
    pass


class InputValidationError(CreditSystemError):
    """Raised when user input violates validation or business rules."""
    pass

# FEATURE PIPELINE & DATA HANDLER CLASS
class CreditDataHandler:
    """Handles raw data ingestion, outlier removal, and feature engineering."""

    @staticmethod
    def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
        """Applies domain financial ratio computations."""
        data = df.copy()
        data["estimated_annual_debt"] = data["loan_amnt"] * (1 + (data["loan_int_rate"] / 100.0))
        data["cred_hist_to_age_ratio"] = data["cb_person_cred_hist_length"] / (data["person_age"] + 1e-5)
        data["emp_to_age_ratio"] = data["person_emp_length"] / (data["person_age"] + 1e-5)
        return data

    def load_and_prepare(self, filepath: str) -> Tuple[pd.DataFrame, pd.Series]:
        """Loads dataset and performs sanity cleaning and transformations."""
        if not os.path.exists(filepath):
            raise DatasetError(f"Dataset file not found at path: '{filepath}'")

        try:
            df = pd.read_csv(filepath)
        except Exception as e:
            raise DatasetError(f"Failed to read CSV: {e}")

        required_cols = {
            "person_age", "person_income", "person_emp_length", "loan_amnt",
            "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length",
            "person_home_ownership", "loan_intent", "loan_grade",
            "cb_person_default_on_file", "loan_status"
        }

        missing = required_cols - set(df.columns)
        if missing:
            raise DatasetError(f"Missing required columns in dataset: {missing}")

        # Remove impossible outlier entries
        df = df[df["person_age"] <= 100]
        df = df[(df["person_emp_length"].isna()) | (df["person_emp_length"] <= 60)]

        if df.empty:
            raise DatasetError("Dataset is empty after removing invalid outliers.")

        df = self.feature_engineering(df)
        X = df.drop(columns=["loan_status"])
        y = df["loan_status"]

        return X, y

# CREDIT SCORING MODEL PIPELINE CLASS
class CreditScoringModel:
    """Encapsulates model definition, preprocessing pipelines, training, and inference."""

    def __init__(self):
        self.numeric_features = [
            "person_age",
            "person_income",
            "person_emp_length",
            "loan_amnt",
            "loan_int_rate",
            "loan_percent_income",
            "cb_person_cred_hist_length",
            "estimated_annual_debt",
            "cred_hist_to_age_ratio",
            "emp_to_age_ratio",
        ]
        self.categorical_features = [
            "person_home_ownership",
            "loan_intent",
            "loan_grade",
            "cb_person_default_on_file",
        ]
        self._pipeline = self._create_pipeline()
        self._is_trained = False

    def _create_pipeline(self) -> Pipeline:
        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                    self.numeric_features,
                ),
                (
                    "cat",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="most_frequent")),
                            ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
                        ]
                    ),
                    self.categorical_features,
                ),
            ]
        )

        classifier = RandomForestClassifier(
            n_estimators=150,
            max_depth=12,
            random_state=42,
            class_weight="balanced"
        )

        return Pipeline(steps=[("preprocessor", preprocessor), ("classifier", classifier)])

    def train(self, X: pd.DataFrame, y: pd.Series) -> None:
        """Trains the scikit-learn pipeline."""
        try:
            self._pipeline.fit(X, y)
            self._is_trained = True
        except Exception as e:
            raise CreditSystemError(f"Model training failed: {e}")

    def predict(self, applicant_data: pd.DataFrame) -> Tuple[int, float]:
        """Returns binary prediction and risk probability."""
        if not self._is_trained:
            raise ModelNotFittedError("Cannot predict: Model pipeline has not been trained yet.")

        try:
            prediction = int(self._pipeline.predict(applicant_data)[0])
            prob_default = float(self._pipeline.predict_proba(applicant_data)[0][1])
            return prediction, prob_default
        except Exception as e:
            raise CreditSystemError(f"Prediction inference error: {e}")


# USER APPLICATION INTERFACE & VALIDATION CLASS
class LoanApplicationCLI:
    """Manages CLI inputs, re-prompting on errors, and displaying reports."""

    VALID_HOME_OWNERSHIP = {"RENT", "OWN", "MORTGAGE", "OTHER"}
    VALID_LOAN_INTENTS = {"PERSONAL", "EDUCATION", "MEDICAL", "VENTURE", "HOMEIMPROVEMENT", "DEBTCONSOLIDATION"}
    VALID_LOAN_GRADES = {"A", "B", "C", "D", "E", "F", "G"}
    VALID_YES_NO = {"Y", "N"}

    @staticmethod
    def _read_float(prompt: str, min_val: float, max_val: float) -> float:
        """Helper to get a valid float within bounds, re-prompting on invalid entry."""
        while True:
            raw = input(prompt).strip()
            try:
                val = float(raw)
                if not (min_val <= val <= max_val):
                    print(f"  [!] Value must be between {min_val} and {max_val}. Try again.")
                    continue
                return val
            except ValueError:
                print("  [!] Invalid numeric format. Please enter a valid number.")

    @staticmethod
    def _read_choice(prompt: str, allowed_choices: set) -> str:
        """Helper to validate dropdown/set choices."""
        while True:
            choice = input(prompt).strip().upper()
            if choice in allowed_choices:
                return choice
            print(f"  [!] Invalid choice '{choice}'. Allowed choices: {', '.join(sorted(allowed_choices))}")

    def collect_applicant_data(self) -> pd.DataFrame:
        """Interactively collects and verifies applicant inputs."""
        print("\n" + "=" * 60)
        print("ENTER APPLICANT FINANCIAL & LOAN DETAILS")
        print("=" * 60)

        # Demographics & Base Financials
        age = self._read_float("Applicant Age (18 - 100): ", 18.0, 100.0)
        income = self._read_float("Annual Income in USD (min 1000): ", 1000.0, 50000000.0)
        emp_length = self._read_float("Employment Length in years (0 - 60): ", 0.0, 60.0)

        if emp_length > (age - 14):
            print(f"  [Note] Employment adjusted: applicant could not have worked before age 14.")
            emp_length = min(emp_length, max(0.0, age - 14))

        home_ownership = self._read_choice(
            "Home Ownership [RENT, OWN, MORTGAGE, OTHER]: ",
            self.VALID_HOME_OWNERSHIP
        )

        print("\nAvailable Intents: PERSONAL, EDUCATION, MEDICAL, VENTURE, HOMEIMPROVEMENT, DEBTCONSOLIDATION")
        loan_intent = self._read_choice("Loan Intent: ", self.VALID_LOAN_INTENTS)
        loan_grade = self._read_choice("Loan Grade [A, B, C, D, E, F, G]: ", self.VALID_LOAN_GRADES)

        loan_amnt = self._read_float("Loan Amount Requested in USD (min 500): ", 500.0, 2000000.0)
        loan_int_rate = self._read_float("Interest Rate in % (0.1 - 50.0): ", 0.1, 50.0)

        # Ratio calculation
        loan_percent_income = round(loan_amnt / income, 2)
        print(f"Calculated Loan-to-Income Percent: {loan_percent_income * 100:.1f}%")

        cb_default = self._read_choice("Prior Default on File? [Y/N]: ", self.VALID_YES_NO)
        cred_hist_length = int(self._read_float("Credit History Length in years (0 - 60): ", 0, min(age - 18, 60)))

        # Assemble into raw single-row DataFrame
        raw_df = pd.DataFrame([{
            "person_age": age,
            "person_income": income,
            "person_home_ownership": home_ownership,
            "person_emp_length": emp_length,
            "loan_intent": loan_intent,
            "loan_grade": loan_grade,
            "loan_amnt": loan_amnt,
            "loan_int_rate": loan_int_rate,
            "loan_percent_income": loan_percent_income,
            "cb_person_default_on_file": cb_default,
            "cb_person_cred_hist_length": cred_hist_length,
        }])

        return CreditDataHandler.feature_engineering(raw_df)

    def display_report(self, prediction: int, prob_default: float) -> None:
        """Renders credit decision report."""
        if prob_default < 0.25:
            risk_tier = "Tier 1: Low Risk (Instant Approval)"
        elif prob_default < 0.50:
            risk_tier = "Tier 2: Moderate Risk (Manual Underwriting Required)"
        elif prob_default < 0.75:
            risk_tier = "Tier 3: High Risk (Higher Collateral / Co-signer Required)"
        else:
            risk_tier = "Tier 4: Extreme Risk (Reject)"

        status_text = "DEFAULT RISK (HIGH)" if prediction == 1 else "NON-DEFAULT (LOW RISK)"

        print("\n" + "=" * 60)
        print("CREDIT ASSESSMENT REPORT")
        print("=" * 60)
        print(f"Final Model Decision  : {status_text}")
        print(f"Probability of Default: {prob_default * 100:.2f}%")
        print(f"Risk Tier Band        : {risk_tier}")
        print("=" * 60 + "\n")


# CONTROLLER / ENTRYPOINT
class CreditAssessmentApp:
    """Orchestrates end-to-end execution."""

    def __init__(self, data_path: str = "credit_risk_dataset.csv"):
        self.data_path = data_path
        self.data_handler = CreditDataHandler()
        self.model = CreditScoringModel()
        self.cli = LoanApplicationCLI()

    def run(self):
        try:
            print("Initializing Credit Scoring System...")
            print(f"Loading data from '{self.data_path}'...")
            X, y = self.data_handler.load_and_prepare(self.data_path)

            print("Fitting Random Forest classification pipeline...")
            self.model.train(X, y)
            print("Model successfully trained and ready for scoring.\n")

            while True:
                applicant_data = self.cli.collect_applicant_data()
                prediction, proba = self.model.predict(applicant_data)
                self.cli.display_report(prediction, proba)

                cont = input("Score another applicant? [Y/n]: ").strip().lower()
                if cont not in ("", "y", "yes"):
                    print("Exiting credit scoring application. Goodbye!")
                    break

        except DatasetError as de:
            print(f"\n[FATAL DATASET ERROR]: {de}", file=sys.stderr)
        except CreditSystemError as cse:
            print(f"\n[SYSTEM ERROR]: {cse}", file=sys.stderr)
        except KeyboardInterrupt:
            print("\nProcess interrupted by user. Exiting.")
        except Exception as ex:
            print(f"\n[UNEXPECTED CRITICAL ERROR]: {ex}", file=sys.stderr)


if __name__ == "__main__":
    app = CreditAssessmentApp(data_path="Credit_Risk_Dataset.csv")
    app.run()

Initializing Credit Scoring System...
Loading data from 'Credit_Risk_Dataset.csv'...
Fitting Random Forest classification pipeline...
Model successfully trained and ready for scoring.


ENTER APPLICANT FINANCIAL & LOAN DETAILS


Applicant Age (18 - 100):  25
Annual Income in USD (min 1000):  40000
Employment Length in years (0 - 60):  25


  [Note] Employment adjusted: applicant could not have worked before age 14.


Home Ownership [RENT, OWN, MORTGAGE, OTHER]:  own



Available Intents: PERSONAL, EDUCATION, MEDICAL, VENTURE, HOMEIMPROVEMENT, DEBTCONSOLIDATION


...

Calculated Loan-to-Income Percent: 1950.0%


Prior Default on File? [Y/N]:  Y
Credit History Length in years (0 - 60):  0



CREDIT ASSESSMENT REPORT
Final Model Decision  : NON-DEFAULT (LOW RISK)
Probability of Default: 30.00%
Risk Tier Band        : Tier 2: Moderate Risk (Manual Underwriting Required)



Score another applicant? [Y/n]:  n


Exiting credit scoring application. Goodbye!
